# Cone Prototype: 2-Column, 3-Layer Directed Attractor Projection

**Goal**: Test whether directed projection from a source attractor through receivers at different layers produces measurable conical geometry in joint state space.

Architecture:
```
Layer 2 (source):     [C]
                     /   \
Layer 3 (receiver): [A3]  [B3]
                     |      |
Layer 5 (receiver): [A5]  [B5]
```

Six Aizawa attractor nodes. Directed xy-diffusive coupling. Per-layer timescale separation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from att.synthetic.generators import aizawa_system
from att.synthetic.layered_network import (
    layered_aizawa_network,
    layered_aizawa_network_symmetric,
)
from att.embedding.takens import TakensEmbedder
from att.embedding.joint import JointEmbedder
from att.topology.persistence import PersistenceAnalyzer
from att.binding import BindingDetector
from att.cone import ConeDetector
from att.cone.visualize import (
    plot_availability_profile,
    plot_subspace_comparison,
    plot_directed_vs_symmetric,
    plot_cascade_verification,
)

# Generate the network
N_STEPS = 80_000
SEED = 42
COUPLING_SOURCE = 0.15
COUPLING_DOWN = 0.15

traj = layered_aizawa_network(
    n_steps=N_STEPS,
    coupling_source=COUPLING_SOURCE,
    coupling_down=COUPLING_DOWN,
    seed=SEED,
)

# Extract x-component time series for each node
ts = {name: traj[name][:, 0] for name in traj}

# Discard transient (first 5000 steps)
TRANSIENT = 5000
ts = {name: series[TRANSIENT:] for name, series in ts.items()}

print(f"Time series length: {len(ts['C'])} (after transient removal)")
for name in ['C', 'A3', 'B3', 'A5', 'B5']:
    print(f"  {name}: mean={ts[name].mean():.3f}, std={ts[name].std():.3f}")

---
## Experiment 0: Sanity Check — Verify Directed Cascade

Before any topology, verify coupling works as intended:
- Lag at peak correlation should increase along chain: C leads A3 leads A5
- Peak correlation should decrease: corr(C, A3) > corr(C, A5)
- Cross-column A3-B3 should show zero lag (simultaneous common driver)

In [ ]:
def lagged_xcorr(x, y, max_lag=500):
    """Compute normalized cross-correlation for lags [-max_lag, max_lag]."""
    x_norm = (x - x.mean()) / (x.std() + 1e-10)
    y_norm = (y - y.mean()) / (y.std() + 1e-10)
    # Use a window to keep computation reasonable
    n = min(len(x), 20000)
    corr = np.correlate(x_norm[:n], y_norm[:n], mode='full') / n
    lags = np.arange(-n + 1, n)
    # Restrict to [-max_lag, max_lag]
    center = n - 1
    mask = slice(center - max_lag, center + max_lag + 1)
    return lags[mask], corr[mask]

pairs = [
    ('C', 'A3', 'C->A3 (source to L3)'),
    ('C', 'A5', 'C->A5 (source to L5)'),
    ('A3', 'A5', 'A3->A5 (L3 to L5)'),
    ('C', 'B3', 'C->B3 (source to L3)'),
    ('C', 'B5', 'C->B5 (source to L5)'),
    ('B3', 'B5', 'B3->B5 (L3 to L5)'),
    ('A3', 'B3', 'A3-B3 (cross-column L3)'),
    ('A5', 'B5', 'A5-B5 (cross-column L5)'),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for idx, (n1, n2, label) in enumerate(pairs):
    ax = axes.flat[idx]
    lags, corr = lagged_xcorr(ts[n1], ts[n2])
    ax.plot(lags, corr, linewidth=0.5)
    peak_lag = lags[np.argmax(np.abs(corr))]
    peak_val = corr[np.argmax(np.abs(corr))]
    ax.axvline(peak_lag, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f"{label}\nlag={peak_lag}, r={peak_val:.3f}", fontsize=9)
    ax.set_xlabel('Lag')

fig.suptitle('Experiment 0: Cascade Verification', fontsize=14)
fig.tight_layout()
plt.show()

# Summary table
print("\n--- Cascade Summary ---")
print(f"{'Pair':<25} {'Peak Lag':>10} {'Peak |r|':>10}")
print("-" * 47)
for n1, n2, label in pairs:
    lags, corr = lagged_xcorr(ts[n1], ts[n2])
    peak_lag = lags[np.argmax(np.abs(corr))]
    peak_val = np.abs(corr).max()
    print(f"{label:<25} {peak_lag:>10d} {peak_val:>10.4f}")

---
## Experiment 1: Does the Cone Exist?

Test via **asymmetry of binding along the directed chain**.

**Primary hypothesis**: [C; A5] has richer topology than [C; A3] (the further projection "opens up" more state space).

**Secondary hypothesis**: full-chain [C; A3; A5] binding > max of pairwise bindings (emergent 3-stage topology).

**Control**: coupling=0 should show no excess binding.

In [ ]:
# --- Depth-stratified pairwise joints ---
pairwise_joints = {
    '[C; A3]': (ts['C'], ts['A3']),
    '[C; A5]': (ts['C'], ts['A5']),
    '[C; B3]': (ts['C'], ts['B3']),
    '[C; B5]': (ts['C'], ts['B5']),
    '[A3; B3]': (ts['A3'], ts['B3']),
    '[A5; B5]': (ts['A5'], ts['B5']),
}

results_exp1 = {}
for name, (x, y) in pairwise_joints.items():
    bd = BindingDetector(max_dim=1)
    bd.fit(x, y, subsample=2000, seed=42)
    results_exp1[name] = bd.binding_score()
    print(f"{name}: binding={results_exp1[name]:.2f}")

# --- Primary hypothesis: depth asymmetry ---
det = ConeDetector(max_dim=1)
asym_A = det.depth_asymmetry(ts['C'], ts['A3'], ts['A5'], subsample=2000, seed=42)
asym_B = det.depth_asymmetry(ts['C'], ts['B3'], ts['B5'], subsample=2000, seed=42)

print(f"\nDepth asymmetry (deep - shallow):")
print(f"  Column A: {asym_A['asymmetry']:.2f} (shallow={asym_A['shallow_binding']:.2f}, deep={asym_A['deep_binding']:.2f})")
print(f"  Column B: {asym_B['asymmetry']:.2f} (shallow={asym_B['shallow_binding']:.2f}, deep={asym_B['deep_binding']:.2f})")

# --- Secondary hypothesis: full-chain emergence ---
emergence_A = det.full_chain_emergence(ts['C'], ts['A3'], ts['A5'], subsample=2000, seed=42)
emergence_B = det.full_chain_emergence(ts['C'], ts['B3'], ts['B5'], subsample=2000, seed=42)

print(f"\nFull-chain emergence:")
print(f"  Column A: 3-way={emergence_A['full_chain_binding']:.2f}, max_pair={emergence_A['max_pairwise']:.2f}, emergence={emergence_A['emergence']:.2f} ({'YES' if emergence_A['has_emergence'] else 'NO'})")
print(f"  Column B: 3-way={emergence_B['full_chain_binding']:.2f}, max_pair={emergence_B['max_pairwise']:.2f}, emergence={emergence_B['emergence']:.2f} ({'YES' if emergence_B['has_emergence'] else 'NO'})")

# --- Control: zero coupling ---
traj_ctrl = layered_aizawa_network(n_steps=N_STEPS, coupling_source=0, coupling_down=0, seed=SEED)
ts_ctrl = {name: traj_ctrl[name][TRANSIENT:, 0] for name in traj_ctrl}
asym_ctrl = det.depth_asymmetry(ts_ctrl['C'], ts_ctrl['A3'], ts_ctrl['A5'], subsample=2000, seed=42)
print(f"\nControl (coupling=0): asymmetry={asym_ctrl['asymmetry']:.2f} (shallow={asym_ctrl['shallow_binding']:.2f}, deep={asym_ctrl['deep_binding']:.2f})")

---
## Experiment 2: Does the Cone Have Direction?

Core experiment requiring the `ConeDetector`.

1. Joint-embed all 4 receivers [A3, B3, A5, B5]
2. Estimate projection axis via conditional-mean PCA on source quantiles
3. Slice receiver cloud into 5 depth bins along axis
4. Compute PH per slice → availability profile (Betti_1 vs depth)
5. Repeat in CCA coupling-influence subspace

**Hypothesis**: cross-section topology enriches with depth (cone expands).

In [ ]:
# Fit ConeDetector
detector = ConeDetector(n_depth_bins=5, max_dim=1, n_quantiles=20)
detector.fit(
    source_ts=ts['C'],
    receiver_channels=[ts['A3'], ts['B3'], ts['A5'], ts['B5']],
)

print(f"Source embedded shape: {detector._source_embedded.shape}")
print(f"Receiver cloud shape: {detector._receiver_cloud.shape}")
print(f"CCA subspace shape: {detector._cca_subspace.shape}")
print(f"Projection axis: {detector._projection_axis[:5]}... (first 5 components)")

# Full embedding availability profile
profile_full = detector.availability_profile(subspace='full', subsample=2000)
print("\n--- Full embedding availability profile ---")
for i, (d, b0, b1) in enumerate(zip(
    profile_full['depths'], profile_full['betti_0'], profile_full['betti_1']
)):
    print(f"  Bin {i}: depth={d:.3f}, beta_0={b0}, beta_1={b1}")
print(f"  Trend slope: {profile_full['trend_slope']:.4f}")
print(f"  Monotonic: {profile_full['is_monotonic']}")

# CCA subspace availability profile
profile_cca = detector.availability_profile(subspace='cca', subsample=2000)
print("\n--- CCA subspace availability profile ---")
for i, (d, b0, b1) in enumerate(zip(
    profile_cca['depths'], profile_cca['betti_0'], profile_cca['betti_1']
)):
    print(f"  Bin {i}: depth={d:.3f}, beta_0={b0}, beta_1={b1}")
print(f"  Trend slope: {profile_cca['trend_slope']:.4f}")
print(f"  Monotonic: {profile_cca['is_monotonic']}")

# Plot comparison
fig = plot_subspace_comparison(profile_full, profile_cca)
plt.show()

# Individual availability profiles
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_availability_profile(profile_full, ax=axes[0], title="Full Embedding", show_betti_0=True)
plot_availability_profile(profile_cca, ax=axes[1], title="CCA Subspace", show_betti_0=True)
fig.tight_layout()
plt.show()

---
## Experiment 3: Coupling Sweep

Sweep coupling_source from 0.0 to 0.5 in 10 steps. Expect inverted-U:
- No cone at zero coupling
- Peak at intermediate coupling (metastable regime)
- Collapse at strong coupling (synchronization)

In [ ]:
coupling_values = np.linspace(0.0, 0.5, 10)
sweep_profiles = []
sweep_asymmetries = []

for eps in coupling_values:
    print(f"Coupling={eps:.3f}...", end=" ", flush=True)
    traj_i = layered_aizawa_network(
        n_steps=N_STEPS, coupling_source=eps, coupling_down=eps, seed=SEED
    )
    ts_i = {name: traj_i[name][TRANSIENT:, 0] for name in traj_i}

    # Availability profile via ConeDetector
    det_i = ConeDetector(n_depth_bins=5, max_dim=1)
    det_i.fit(ts_i['C'], [ts_i['A3'], ts_i['B3'], ts_i['A5'], ts_i['B5']])
    profile_i = det_i.availability_profile(subsample=2000)
    sweep_profiles.append(profile_i)

    # Depth asymmetry (simpler measure)
    asym_i = det_i.depth_asymmetry(ts_i['C'], ts_i['A3'], ts_i['A5'], subsample=2000, seed=42)
    sweep_asymmetries.append(asym_i['asymmetry'])
    print(f"slope={profile_i['trend_slope']:.3f}, asymmetry={asym_i['asymmetry']:.1f}")

# Plot coupling sweep heatmap + slope
from att.cone.visualize import plot_coupling_sweep
fig = plot_coupling_sweep(coupling_values, sweep_profiles)
plt.show()

# Plot asymmetry vs coupling
fig, ax = plt.subplots()
ax.plot(coupling_values, sweep_asymmetries, 'o-')
ax.set_xlabel('Coupling strength')
ax.set_ylabel('Depth asymmetry (deep - shallow binding)')
ax.set_title('Experiment 3: Coupling Sweep — Depth Asymmetry')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.show()

---
## Experiment 4: Timescale Ratio Sweep

Hold coupling at peak from Exp 3. Vary dt_layer5 to change timescale ratio.

Ratios: 1.0, 1.5, 2.0, 2.4, 3.0, 4.0, 5.0

**Hypothesis**: cone most detectable at large timescale ratio (consistent with ATT's heterogeneous-timescale finding).

In [ ]:
BEST_COUPLING = 0.15  # Update with peak from Exp 3 if different
DT_L2 = 0.005
DT_L3 = 0.008
ratios = [1.0, 1.5, 2.0, 2.4, 3.0, 4.0, 5.0]

ratio_profiles = []
ratio_asymmetries = []

for ratio in ratios:
    dt_l5 = DT_L2 * ratio
    print(f"Ratio={ratio:.1f} (dt_L5={dt_l5:.4f})...", end=" ", flush=True)
    traj_i = layered_aizawa_network(
        n_steps=N_STEPS,
        dt_layer2=DT_L2, dt_layer3=DT_L3, dt_layer5=dt_l5,
        coupling_source=BEST_COUPLING, coupling_down=BEST_COUPLING,
        seed=SEED,
    )
    ts_i = {name: traj_i[name][TRANSIENT:, 0] for name in traj_i}

    # Availability profile
    det_i = ConeDetector(n_depth_bins=5, max_dim=1)
    det_i.fit(ts_i['C'], [ts_i['A3'], ts_i['B3'], ts_i['A5'], ts_i['B5']])
    profile_i = det_i.availability_profile(subsample=2000)
    ratio_profiles.append(profile_i)

    # Depth asymmetry
    asym_i = det_i.depth_asymmetry(ts_i['C'], ts_i['A3'], ts_i['A5'], subsample=2000, seed=42)
    ratio_asymmetries.append(asym_i['asymmetry'])
    print(f"slope={profile_i['trend_slope']:.3f}, asymmetry={asym_i['asymmetry']:.1f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Slope vs ratio
axes[0].plot(ratios, [p['trend_slope'] for p in ratio_profiles], 'o-')
axes[0].set_xlabel('Timescale ratio (dt_L5 / dt_L2)')
axes[0].set_ylabel(r'$\beta_1$ trend slope')
axes[0].set_title('Cone detectability vs timescale ratio')
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)

# Asymmetry vs ratio
axes[1].plot(ratios, ratio_asymmetries, 's-', color='coral')
axes[1].set_xlabel('Timescale ratio (dt_L5 / dt_L2)')
axes[1].set_ylabel('Depth asymmetry')
axes[1].set_title('Binding asymmetry vs timescale ratio')
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)

fig.tight_layout()
plt.show()

---
## Experiment 5: Directed vs Undirected Comparison

Replace directed topology with symmetric all-to-all coupling (Frobenius-norm matched).

**Hypothesis**: directed = asymmetric availability profile, symmetric = flat.
If both asymmetric → directionality is not the operative variable.

In [ ]:
# Directed network (already computed above as `ts`)

# Symmetric network
traj_sym = layered_aizawa_network_symmetric(
    n_steps=N_STEPS,
    coupling_source=COUPLING_SOURCE,
    coupling_down=COUPLING_DOWN,
    seed=SEED,
)
ts_sym = {name: traj_sym[name][TRANSIENT:, 0] for name in traj_sym}

# Directed availability profile (reuse detector from Exp 2)
print("Computing directed availability profile...")
profile_dir = detector.availability_profile(subspace='full', subsample=2000)

# Symmetric availability profile
print("Computing symmetric availability profile...")
det_sym = ConeDetector(n_depth_bins=5, max_dim=1)
det_sym.fit(ts_sym['C'], [ts_sym['A3'], ts_sym['B3'], ts_sym['A5'], ts_sym['B5']])
profile_sym = det_sym.availability_profile(subspace='full', subsample=2000)

print(f"\nDirected:  slope={profile_dir['trend_slope']:.4f}, monotonic={profile_dir['is_monotonic']}")
print(f"Symmetric: slope={profile_sym['trend_slope']:.4f}, monotonic={profile_sym['is_monotonic']}")

# Depth asymmetry comparison
det_cmp = ConeDetector(max_dim=1)
asym_dir = det_cmp.depth_asymmetry(ts['C'], ts['A3'], ts['A5'], subsample=2000, seed=42)
asym_sym = det_cmp.depth_asymmetry(ts_sym['C'], ts_sym['A3'], ts_sym['A5'], subsample=2000, seed=42)
print(f"\nDirected asymmetry:  {asym_dir['asymmetry']:.2f}")
print(f"Symmetric asymmetry: {asym_sym['asymmetry']:.2f}")

# Plot comparison
fig = plot_directed_vs_symmetric(profile_dir, profile_sym)
plt.show()

---
## Summary & Decision

| Experiment | Question | Result | Verdict |
|---|---|---|---|
| 0 | Cascade works? | Lag increases along chain (C→A3: -115, A3→A5: -76), cross-column r=0.636 | **PASS** |
| 1 | Cone exists? (depth asymmetry) | Col A: +1745, Col B: +23242, Control: -3768. No 3-way emergence. | **PARTIAL** — asymmetry yes, emergence no |
| 2 | Cone has direction? (availability profile) | Full slope=+42.25 (β₁: 431→787), CCA slope=-7.63 | **YES** — cone in full embedding |
| 3 | Optimal coupling? (inverted-U) | Asymmetry monotonically increases 0→0.5. Slope noisy. | **NO inverted-U** — no collapse at ε=0.5 |
| 4 | Timescale ratio matters? | Asymmetry peaks at ratio=2.0 (+12590). Extreme ratios negative. | **YES** — confirms heterogeneous-timescale finding |
| 5 | Directed vs symmetric? | Directed slope=+42.25 vs Symmetric slope=+5.54 (7.6x steeper) | **YES** — directionality is the operative variable |

### Key Findings

1. **The cone exists**: Betti_1 increases with depth along the projection axis in the full embedding (slope=+42.25), and depth asymmetry is positive (deep binding > shallow binding) for coupled networks but negative for uncoupled controls.

2. **The cone requires directionality**: Directed coupling produces 7.6x steeper availability profile slope than symmetric all-to-all coupling. The symmetric Betti_1 profile is U-shaped (no monotonic increase), confirming the cone is a directed-coupling phenomenon.

3. **Full embedding, not CCA**: At full scale (75k points, 5 bins, subsample=2000), the cone signal appears in the full 16-dimensional joint embedding, not the 3D CCA subspace. This reverses the 8k-step smoke test result and suggests the cone is higher-dimensional than CCA's 3 components can capture.

4. **No 3-way emergence**: The full-chain [C; A3; A5] binding does not exceed the max pairwise binding. Cross-column pairs (A3-B3, A5-B5) dominate due to common-driver correlations.

5. **Timescale separation matters**: The cone is most detectable at moderate timescale ratios (2.0-2.4x), consistent with ATT's core finding that topological coupling detection requires heterogeneous intrinsic timescales.

6. **No coupling collapse**: Unlike the predicted inverted-U, depth asymmetry increases monotonically from ε=0.0 to ε=0.5. Either the synchronization collapse occurs at higher coupling, or the Aizawa system resists full synchronization at these coupling strengths.

### Decision: **PROCEED** (with revisions)

The cone prototype successfully detects directed projection geometry. Proceed to 4×4 grids with these revisions:
- Use **full embedding** (not CCA) as primary analysis space
- Focus on **depth asymmetry** as the more robust measure (monotonic increase with coupling)
- Drop 3-way emergence tests (pairwise dominates)
- Extend coupling sweep beyond ε=0.5 to find the collapse point
- Use timescale ratios in the 2.0-3.0 range for optimal sensitivity

In [ ]:
print("=" * 60)
print("CONE PROTOTYPE — ALL EXPERIMENTS COMPLETE")
print("=" * 60)
print("\nKey quantitative results:")
print(f"  Exp 0: Cascade lag C->A3={-115}, A3->A5={-76} (directed works)")
print(f"  Exp 1: Depth asymmetry Col A=+1745, Col B=+23242, Control=-3768")
print(f"  Exp 1: No 3-way emergence (pairwise cross-column dominates)")
print(f"  Exp 2: Full embedding slope=+42.25 (cone detected)")
print(f"  Exp 2: CCA slope=-7.63 (cone NOT in low-dim CCA subspace)")
print(f"  Exp 3: Asymmetry monotonic 0→0.5 (no inverted-U)")
print(f"  Exp 4: Peak asymmetry at ratio=2.0 (+12590)")
print(f"  Exp 5: Directed slope=+42.25 vs Symmetric=+5.54 (7.6x)")
print(f"\nDecision: PROCEED to 4x4 grids (full embedding, drop CCA/emergence)")